#로보플로우 라이브러리 설치

In [1]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 77.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18


로보플로우 데이터셋 다운로드

In [2]:
from roboflow import Roboflow

rf = Roboflow(api_key="hvf5xccIgIHr180mC45H")
project = rf.workspace("huangzixuan").project("drugs-detection")
version = project.version(3)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to drugs-detection-3 in yolov8:: 100%|██████████| 5392/5392 [00:01<00:00, 3504.12it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


#Model 및 학습

In [3]:
from __future__ import annotations

import argparse
from pathlib import Path

from ultralytics import YOLO


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train YOLOv8 image detection model")
    parser.add_argument("--data", type=str, default="/content/drugs-detection-3/data.yaml", help="data.yaml path")
    parser.add_argument("--model", type=str, default="yolov8n.pt", help="base model: yolov8n.pt/yolov8s.pt/yolov8m.pt")
    parser.add_argument("--epochs", type=int, default=50, help="training epochs")
    parser.add_argument("--imgsz", type=int, default=640, help="input image size")
    parser.add_argument("--batch", type=int, default=16, help="batch size")
    parser.add_argument("--device", type=str, default="0", help="CUDA device id, cpu, or mps")
    parser.add_argument("--project", type=str, default="/content/runs/detect", help="Ultralytics output project dir")
    parser.add_argument("--name", type=str, default="drug_yolov8n_final", help="experiment name")
    parser.add_argument("--patience", type=int, default=20, help="early stopping patience")
    parser.add_argument("--seed", type=int, default=42, help="random seed")
    parser.add_argument("--workers", type=int, default=8, help="dataloader workers")

    args, _ = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_args()
    data_path = Path(args.data)
    if not data_path.exists():
        raise FileNotFoundError(f"data.yaml을 찾을 수 없습니다: {data_path}")

    model = YOLO(args.model)
    model.train(
        data=str(data_path),
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        device=args.device,
        project=args.project,
        name=args.name,
        patience=args.patience,
        seed=args.seed,
        workers=args.workers,
        plots=True,
    )


if __name__ == "__main__":
    main()


Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drugs-detection-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=drug_yolov8n_final, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

#Evaluation
- Precision, Recall, mAP50, mAP50-95
- Class별 평가지수

In [5]:
from ultralytics import YOLO

WEIGHTS = "/content/runs/detect/drug_yolov8n_final/weights/best.pt"
DATA = "/content/drugs-detection-3/data.yaml"

model = YOLO(WEIGHTS)

metrics = model.val(
    data=DATA,
    split="test",
    plots=True
)

print("\n===== YOLOv8 Test Evaluation =====")
print(f"Precision : {metrics.box.mp:.6f}")
print(f"Recall    : {metrics.box.mr:.6f}")
print(f"mAP50     : {metrics.box.map50:.6f}")
print(f"mAP50-95  : {metrics.box.map:.6f}")

print("\n===== Class-wise Results =====")
for class_id, class_name in metrics.names.items():
    p, r, ap50, ap = metrics.box.class_result(class_id)

    print(f"\n[{class_id}] {class_name}")
    print(f"Precision : {p:.6f}")
    print(f"Recall    : {r:.6f}")
    print(f"mAP50     : {ap50:.6f}")
    print(f"mAP50-95  : {ap:.6f}")

Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2501.7±970.5 MB/s, size: 60.4 KB)
val: Scanning /content/drugs-detection-3/test/labels.cache... 139 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 139/139 64.8Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 36, len(boxes) = 290. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.1it/s 4.3s
                   all        139        290      0.929      0.877      0.908       0.67
               Cocaine         58         73      0.972      0.963      0.991      0.771
                Heroin         28     